In [5]:
pip install customtkinter

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
pip install ultralytics

  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ---------------------------------------- 0.0/1.3 MB ? eta -:--:--
   ---------------------------------------- 1.3/1.3 MB 11.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/833.4 kB ? eta -:--:--
   ---------------------------------------- 833.4/833.4 kB 9.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/52.0 MB ? eta -:--:--
   -- ------------------------------------- 2.9/52.0 MB 12.9 MB/s eta 0:00:04
   ---- ----------------------------------- 5.5/52.0 MB 13.4 MB/s eta 0:00:04
   ------ --------------------------------- 8.4/52.0 MB 13.3 MB/s eta 0:00:04
   -------- ------------------------------- 11.5/52.0 MB 13.6 MB/s eta 0:00:03
   ----------- ---------------------------- 14.4/52.0 MB 13.7 MB/s eta 0:00:03
   ------------- -------------------------- 17.6/52


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import customtkinter as ctk
import cv2
from PIL import Image
import threading
import queue
import time
from ultralytics import YOLO
import struct
import numpy as np
import socket

# --- Configuration ---
STREAM_URL = "http://192.168.0.7:8080/video"
MODEL_PATH = r"runs\detect\phase1_yolo26s_results\weights\best.pt"
TARGET_CLASSES = ["bottle", "laptop"]

CONFIDENCE_THRESHOLD = 0.45
NMS_IOU_THRESHOLD = 0.45
SKIP_FRAMES = 7
MIN_BOX_AREA_FRACTION = 0.01

ANNOUNCE_COOLDOWN_SEC = 8.0
POSITION_CHANGE_COOLDOWN = 0.5

# --- Per-class distance thresholds (fraction of frame area) ---
DISTANCE_THRESHOLDS = {
    "bottle": {"close": 0.06, "nearby": 0.02},
    "laptop": {"close": 0.18, "nearby": 0.06},
}
DEFAULT_DISTANCE_THRESHOLDS = {"close": 0.10, "nearby": 0.03}

# --- Announcement Sender (audio now plays through glasses) ---
announcement_socket = None
announcement_lock = threading.Lock()

def connect_announcement_socket():
    """Maintains a persistent listener on port 9998 for the phone to connect to."""
    global announcement_socket
    while True:
        try:
            sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
            sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
            sock.bind(('0.0.0.0', 9998))
            sock.listen(1)
            print("[Announcer] Waiting for phone on port 9998...")
            conn, addr = sock.accept()
            print(f"[Announcer] Phone connected from {addr}")
            with announcement_lock:
                announcement_socket = conn
            # Hold until connection drops
            while True:
                try:
                    data = conn.recv(1)
                    if not data:
                        break
                except:
                    break
            with announcement_lock:
                announcement_socket = None
            print("[Announcer] Phone disconnected, waiting again...")
            conn.close()
            sock.close()
        except Exception as e:
            print(f"[Announcer] Error: {e}")
            time.sleep(2)

threading.Thread(target=connect_announcement_socket, daemon=True).start()

def speak_async(class_name, quadrant, distance):
    """Sends announcement to phone as: classname|quadrant|distance"""
    with announcement_lock:
        if announcement_socket is None:
            return
        try:
            msg = f"{class_name}|{quadrant}|{distance}".encode('utf-8')
            announcement_socket.sendall(struct.pack('>I', len(msg)) + msg)
        except Exception as e:
            print(f"[Announcer] Send error: {e}")

# --- Distance Estimation (per-class calibrated) ---
def get_distance(class_name, box_area, frame_area):
    fraction = box_area / frame_area
    thresholds = DISTANCE_THRESHOLDS.get(class_name, DEFAULT_DISTANCE_THRESHOLDS)
    if fraction >= thresholds["close"]:
        return "close"
    elif fraction >= thresholds["nearby"]:
        return "nearby"
    else:
        return "far"

# --- Position and Distance Aware Cooldown ---
last_announced = {}

def should_announce(class_name, current_quadrant, current_distance):
    now = time.time()
    if class_name not in last_announced:
        last_announced[class_name] = {
            "time": now,
            "quadrant": current_quadrant,
            "distance": current_distance
        }
        return True

    last = last_announced[class_name]
    time_passed = now - last["time"]
    position_changed = last["quadrant"] != current_quadrant
    distance_changed = last["distance"] != current_distance

    if (position_changed or distance_changed) and time_passed > POSITION_CHANGE_COOLDOWN:
        last_announced[class_name] = {
            "time": now,
            "quadrant": current_quadrant,
            "distance": current_distance
        }
        return True

    if not position_changed and not distance_changed and time_passed > ANNOUNCE_COOLDOWN_SEC:
        last_announced[class_name] = {
            "time": now,
            "quadrant": current_quadrant,
            "distance": current_distance
        }
        return True

    return False

# --- Spatial Grid ---
def get_quadrant(cx, cy, width, height):
    mid_x, mid_y = width // 2, height // 2
    centre_margin_x = width * 0.20
    centre_margin_y = height * 0.20

    if (abs(cx - mid_x) < centre_margin_x and abs(cy - mid_y) < centre_margin_y):
        return "Centre"

    if cx < mid_x and cy < mid_y:
        return "Top Left"
    elif cx >= mid_x and cy < mid_y:
        return "Top Right"
    elif cx < mid_x and cy >= mid_y:
        return "Bottom Left"
    else:
        return "Bottom Right"

# --- Glasses Frame Receiver ---
class GlassesFrameReceiver:
    def __init__(self, port=9999):
        self.port = port
        self.server = None
        self.conn = None

    def start(self):
        self.server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        self.server.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        self.server.bind(('0.0.0.0', self.port))
        self.server.listen(1)
        self.server.settimeout(2.0)  # non-blocking accept
        print(f"[GlassesReceiver] Waiting for Android app on port {self.port}...")
        self._accept()

    def _accept(self):
        while True:
            try:
                self.conn, addr = self.server.accept()
                self.conn.settimeout(5.0)
                print(f"[GlassesReceiver] Connected from {addr}")
                return
            except socket.timeout:
                continue

    def read_frame(self):
        try:
            raw_len = self._recv_exact(4)
            if not raw_len:
                self._reconnect()
                return False, None
            msg_len = struct.unpack('>I', raw_len)[0]
            if msg_len > 10_000_000:
                self._reconnect()
                return False, None
            jpg_bytes = self._recv_exact(msg_len)
            if not jpg_bytes:
                self._reconnect()
                return False, None
            arr = np.frombuffer(jpg_bytes, dtype=np.uint8)
            frame = cv2.imdecode(arr, cv2.IMREAD_COLOR)
            return (frame is not None), frame
        except Exception as e:
            print(f"[GlassesReceiver] Read error: {e}")
            self._reconnect()
            return False, None

    def _reconnect(self):
        print(f"[GlassesReceiver] Connection lost — waiting for reconnect...")
        if self.conn:
            try:
                self.conn.close()
            except:
                pass
            self.conn = None
        self._accept()

    def _recv_exact(self, n):
        data = b''
        while len(data) < n:
            try:
                chunk = self.conn.recv(n - len(data))
                if not chunk:
                    return None
                data += chunk
            except socket.timeout:
                return None
        return data

    def release(self):
        if self.conn:
            self.conn.close()
        if self.server:
            self.server.close()

# --- GUI Application Class ---
class SpatialSyncDashboard(ctk.CTk):
    def __init__(self):
        super().__init__()

        self.title("SpatialSync - Spatial Engine Control Room")
        self.geometry("1200x700")
        ctk.set_appearance_mode("dark")
        ctk.set_default_color_theme("blue")

        self.grid_columnconfigure(0, weight=3)
        self.grid_columnconfigure(1, weight=1)
        self.grid_rowconfigure(0, weight=1)

        # --- Video Panel (Left) ---
        self.video_frame = ctk.CTkFrame(self, corner_radius=10)
        self.video_frame.grid(row=0, column=0, padx=10, pady=10, sticky="nsew")

        self.video_label = ctk.CTkLabel(self.video_frame, text="Waiting for glasses stream...")
        self.video_label.pack(expand=True, fill="both", padx=10, pady=10)

        # --- Control Panel (Right) ---
        self.control_frame = ctk.CTkFrame(self, corner_radius=10)
        self.control_frame.grid(row=0, column=1, padx=(0, 10), pady=10, sticky="nsew")

        self.title_label = ctk.CTkLabel(
            self.control_frame,
            text="SpatialSync",
            font=ctk.CTkFont(size=22, weight="bold")
        )
        self.title_label.pack(pady=(20, 2))

        self.subtitle_label = ctk.CTkLabel(
            self.control_frame,
            text="Assistive Object Detection",
            font=ctk.CTkFont(size=12),
            text_color="gray"
        )
        self.subtitle_label.pack(pady=(0, 12))

        # Live stats bar
        self.stats_frame = ctk.CTkFrame(self.control_frame, corner_radius=8)
        self.stats_frame.pack(fill="x", padx=20, pady=(0, 10))

        self.detection_count_label = ctk.CTkLabel(
            self.stats_frame,
            text="Detections: 0",
            font=ctk.CTkFont(size=12, weight="bold")
        )
        self.detection_count_label.pack(side="left", padx=10, pady=6)

        self.audio_status_label = ctk.CTkLabel(
            self.stats_frame,
            text="Audio: waiting",
            font=ctk.CTkFont(size=12),
            text_color="gray"
        )
        self.audio_status_label.pack(side="right", padx=10, pady=6)

        # Detection log
        self.log_box = ctk.CTkTextbox(
            self.control_frame,
            height=280,
            state="disabled",
            font=ctk.CTkFont(size=12)
        )
        self.log_box.pack(fill="x", padx=20, pady=10)

        # Colour tags for distance
        self.log_box.tag_config("close",  foreground="#FF4C4C")
        self.log_box.tag_config("nearby", foreground="#FFC107")
        self.log_box.tag_config("far",    foreground="#AAAAAA")
        self.log_box.tag_config("system", foreground="#4FC3F7")

        self.log_message("System Initialized.", tag="system")

        # Divider label
        ctk.CTkLabel(
            self.control_frame,
            text="LEGEND",
            font=ctk.CTkFont(size=10),
            text_color="gray"
        ).pack(pady=(4, 2))

        legend_frame = ctk.CTkFrame(self.control_frame, corner_radius=6)
        legend_frame.pack(fill="x", padx=20, pady=(0, 10))

        for label, colour in [("● Close", "#FF4C4C"), ("● Nearby", "#FFC107"), ("● Far", "#AAAAAA")]:
            ctk.CTkLabel(
                legend_frame,
                text=label,
                text_color=colour,
                font=ctk.CTkFont(size=11)
            ).pack(side="left", padx=8, pady=4)

        # Threading state
        self.running = True
        self.latest_rgb_frame = None
        self.current_detection_count = 0

        # Load YOLO model
        try:
            self.model = YOLO(MODEL_PATH)
            self.log_message("YOLO Model Loaded.", tag="system")
        except Exception as e:
            self.log_message(f"ERROR: Could not load model. {e}", tag="system")
            self.model = None

        self.log_message("Starting vision thread...", tag="system")
        self.vision_thread = threading.Thread(target=self.vision_worker, daemon=True)
        self.vision_thread.start()

        self.update_gui_loop()

    def log_message(self, message, tag="system"):
        self.log_box.configure(state="normal")
        self.log_box.insert("end", f"> {message}\n", tag)
        self.log_box.see("end")
        self.log_box.configure(state="disabled")

    def vision_worker(self):
        receiver = GlassesFrameReceiver(port=9999)
        receiver.start()

        frame_count = 0
        last_boxes = []

        while self.running:
            ret, frame = receiver.read_frame()
            if not ret:
                continue

            frame_count += 1
            height, width = frame.shape[:2]
            frame_area = width * height
            cx_margin = int(width * 0.20)
            cy_margin = int(height * 0.20)

            if self.model is not None and frame_count % SKIP_FRAMES == 0:
                results = self.model(
                    frame,
                    conf=CONFIDENCE_THRESHOLD,
                    iou=NMS_IOU_THRESHOLD,
                    imgsz=320,
                    verbose=False
                )
                last_boxes = []

                for result in results:
                    for box in result.boxes:
                        cls = int(box.cls[0])
                        class_name = self.model.names[cls]

                        if class_name in TARGET_CLASSES:
                            x1, y1, x2, y2 = map(int, box.xyxy[0])
                            conf = float(box.conf[0])

                            box_area = (x2 - x1) * (y2 - y1)
                            if box_area < (MIN_BOX_AREA_FRACTION * frame_area):
                                continue

                            cx, cy = (x1 + x2) // 2, (y1 + y2) // 2
                            quadrant = get_quadrant(cx, cy, width, height)
                            distance = get_distance(class_name, box_area, frame_area)
                            conf_pct = int(conf * 100)

                            display_text = f"{class_name} {conf_pct}% | {quadrant} | {distance}"
                            last_boxes.append((x1, y1, x2, y2, display_text, class_name, quadrant, distance))

                            if should_announce(class_name, quadrant, distance):
                                speak_async(class_name, quadrant, distance)
                                self.log_message(
                                    f"{class_name} | {quadrant} | {distance}",
                                    tag=distance
                                )

                with announcement_lock:
                    connected = announcement_socket is not None
                self.audio_status_label.configure(
                    text="Audio: ✓ glasses" if connected else "Audio: waiting",
                    text_color="#4CAF50" if connected else "gray"
                )
                self.current_detection_count = len(last_boxes)

            # Draw bounding boxes
            distance_bgr = {
                "close":  (0, 80, 255),
                "nearby": (0, 200, 255),
                "far":    (180, 180, 180)
            }

            for (x1, y1, x2, y2, display_text, class_name, quadrant, distance) in last_boxes:
                colour = distance_bgr.get(distance, (0, 255, 0))
                cv2.rectangle(frame, (x1, y1), (x2, y2), colour, 2)
                cv2.circle(frame, ((x1 + x2) // 2, (y1 + y2) // 2), 5, (0, 0, 255), -1)
                cv2.putText(frame, display_text, (x1, y1 - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.55, colour, 2)

            # Grid overlay
            cv2.line(frame, (width // 2, 0), (width // 2, height), (255, 0, 0), 1)
            cv2.line(frame, (0, height // 2), (width, height // 2), (255, 0, 0), 1)
            cv2.rectangle(frame,
                        (width // 2 - cx_margin, height // 2 - cy_margin),
                        (width // 2 + cx_margin, height // 2 + cy_margin),
                        (0, 255, 255), 1)
            cv2.putText(frame, "C", (width // 2 - 5, height // 2 + 5),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1)


            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            self.latest_rgb_frame = rgb_frame

        receiver.release()

    def update_gui_loop(self):
        if self.latest_rgb_frame is not None:
            pil_img = Image.fromarray(self.latest_rgb_frame)
            ctk_img = ctk.CTkImage(light_image=pil_img, dark_image=pil_img, size=(750, 530))
            self.video_label.configure(image=ctk_img, text="")

        # Update live detection counter
        self.detection_count_label.configure(
            text=f"Detections: {self.current_detection_count}"
        )

        self.after(30, self.update_gui_loop)

# --- Execution ---
if __name__ == "__main__":
    app = SpatialSyncDashboard()
    app.mainloop()

[Announcer] Waiting for phone on port 9998...
[GlassesReceiver] Waiting for Android app on port 9999...
[Announcer] Phone connected from ('10.64.160.22', 37202)[GlassesReceiver] Connected from ('10.64.160.22', 48460)

[GlassesReceiver] Connection lost — waiting for reconnect...
[GlassesReceiver] Connected from ('10.64.160.22', 44322)
[GlassesReceiver] Connection lost — waiting for reconnect...
[GlassesReceiver] Connected from ('10.64.160.22', 52926)
[GlassesReceiver] Connection lost — waiting for reconnect...
[GlassesReceiver] Connected from ('10.64.160.22', 53928)
[GlassesReceiver] Connection lost — waiting for reconnect...
[GlassesReceiver] Connected from ('10.64.160.22', 47478)
[GlassesReceiver] Connection lost — waiting for reconnect...
[GlassesReceiver] Connected from ('10.64.160.22', 53422)
[GlassesReceiver] Connection lost — waiting for reconnect...
[Announcer] Phone disconnected, waiting again...
[Announcer] Waiting for phone on port 9998...
[GlassesReceiver] Connected from ('1

Exception in thread Thread-12 (vision_worker):
Traceback (most recent call last):
  File "c:\Users\priya\AppData\Local\Programs\Python\Python311\Lib\threading.py", line 1038, in _bootstrap_inner
    self.run()
  File "c:\Users\priya\AppData\Local\Programs\Python\Python311\Lib\threading.py", line 975, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\priya\AppData\Local\Temp\ipykernel_37084\3125830636.py", line 399, in vision_worker
  File "c:\Users\priya\AppData\Local\Programs\Python\Python311\Lib\site-packages\customtkinter\windows\widgets\ctk_label.py", line 206, in configure
    self._label.configure(text=self._text)
  File "c:\Users\priya\AppData\Local\Programs\Python\Python311\Lib\tkinter\__init__.py", line 1702, in configure
    return self._configure('configure', cnf, kw)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\priya\AppData\Local\Programs\Python\Python311\Lib\tkinter\__init__.py", line 1692, in _configure
    self.tk.call(_flatten((

[GlassesReceiver] Connected from ('10.64.160.22', 57184)
[Announcer] Phone disconnected, waiting again...
[Announcer] Waiting for phone on port 9998...


Exception in thread Thread-36 (vision_worker):
Traceback (most recent call last):
  File "c:\Users\priya\AppData\Local\Programs\Python\Python311\Lib\threading.py", line 1038, in _bootstrap_inner
    self.run()
  File "c:\Users\priya\AppData\Local\Programs\Python\Python311\Lib\threading.py", line 975, in run
    self._target(*self._args, **self._kwargs)
  File "C:\Users\priya\AppData\Local\Temp\ipykernel_37084\3125830636.py", line 392, in vision_worker
  File "C:\Users\priya\AppData\Local\Temp\ipykernel_37084\3125830636.py", line 336, in log_message
  File "c:\Users\priya\AppData\Local\Programs\Python\Python311\Lib\site-packages\customtkinter\windows\widgets\ctk_textbox.py", line 305, in configure
    self._textbox.configure(**pop_from_dict_by_set(kwargs, self._valid_tk_text_attributes))
  File "c:\Users\priya\AppData\Local\Programs\Python\Python311\Lib\tkinter\__init__.py", line 1702, in configure
    return self._configure('configure', cnf, kw)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

[GlassesReceiver] Connected from ('10.64.160.22', 36174)
[Announcer] Phone connected from ('10.64.160.22', 35608)
[GlassesReceiver] Connection lost — waiting for reconnect...
[GlassesReceiver] Connected from ('10.64.160.22', 36180)
[GlassesReceiver] Connection lost — waiting for reconnect...
[GlassesReceiver] Connected from ('10.64.160.22', 45942)
[GlassesReceiver] Connection lost — waiting for reconnect...
[GlassesReceiver] Connected from ('10.64.160.22', 45954)
[GlassesReceiver] Connection lost — waiting for reconnect...
[GlassesReceiver] Connected from ('10.64.160.22', 52660)
[GlassesReceiver] Connection lost — waiting for reconnect...
[GlassesReceiver] Connected from ('10.64.160.22', 37386)
[GlassesReceiver] Connection lost — waiting for reconnect...
[GlassesReceiver] Connected from ('10.64.160.22', 42380)
[GlassesReceiver] Connection lost — waiting for reconnect...
[GlassesReceiver] Connected from ('10.64.160.22', 42392)
[GlassesReceiver] Connection lost — waiting for reconnect...